# FPM-INR: Fourier Ptychographic Microscopy with Implicit Neural Representations

## Problem Statement

**Given:** $L = 68$ low-resolution intensity images of a human blood smear, each captured under a different LED illumination angle. The microscope uses a 10x / 0.25 NA objective and a camera with 3.45 um pixel pitch. The sample is tilted approximately 4 degrees relative to the optical axis, creating a 3D reconstruction challenge.

**Goal:** Recover the full 3D complex field $O(x, y, z)$ --- both **amplitude** $A(x, y, z)$ and **phase** $\phi(x, y, z)$ --- at a spatial resolution far exceeding the objective's native numerical aperture.

**Why this is hard:**
1. **Phase is lost.** Cameras measure only intensity $|E|^2$, not the electric field $E$. Recovering the phase from intensity-only measurements is fundamentally ill-posed.
2. **Limited NA per image.** Each individual image has resolution limited by the objective's NA = 0.25, which is far below the synthetic NA achievable by combining all LED angles.
3. **3D volume from 2D measurements.** The sample extends in depth ($z$ from $-20$ to $+20$ $\mu$m), yet each measurement is a 2D image. Depth information must be inferred from defocus-dependent changes across LED angles.
4. **Ill-posed inverse problem.** There are far more unknowns (complex field at every $(x, y, z)$ voxel) than measurements (68 intensity images), requiring strong regularization or learned priors.

**Key imaging parameters:**
- Objective: 10x magnification, 0.25 NA
- Illumination: 68 LEDs at wavelength $\lambda = 512$ nm (green channel)
- Camera pixel pitch: 3.45 $\mu$m
- Sample: blood smear tilted ~4 degrees to the optical axis
- Reconstruction volume: 2048 x 2048 pixels (2x upsampled), 161 z-slices from $-20$ to $+20$ $\mu$m

## Forward Model

Fourier Ptychographic Microscopy (FPM) builds a high-resolution image by combining many low-resolution images taken under different LED illumination angles. Here is the physics, step by step.

### Step 1: Fourier transform of the complex field

The sample is described by a complex-valued field $O(x, y, z) = A(x, y, z) \cdot e^{i\phi(x, y, z)}$, where $A$ is amplitude and $\phi$ is phase. Its 2D spatial Fourier transform at each depth $z$ is:

$$\tilde{O}(k_x, k_y, z) = \mathcal{F}\{O(x, y, z)\}$$

where $k_x, k_y$ are spatial frequency coordinates.

### Step 2: LED illumination shifts the spectrum

Each LED $\ell$ illuminates the sample from a direction characterized by numerical aperture components $(u_\ell, v_\ell)$. This tilted illumination shifts the spatial frequency spectrum so that a different sub-band falls within the objective's pupil $P(k_x, k_y)$. The pupil is a circular aperture in frequency space defined by:

$$P(k_x, k_y) = \begin{cases} 1 & \text{if } k_x^2 + k_y^2 \leq k_{\text{max}}^2 \\ 0 & \text{otherwise} \end{cases}$$

where $k_{\text{max}} = \text{NA} \cdot k_0$ and $k_0 = 2\pi / \lambda$ is the free-space wavenumber.

The sub-band selected by LED $\ell$ is:

$$\tilde{O}_{\text{sub},\ell}(k_x, k_y, z) = \tilde{O}(k_x + k_0 u_\ell,\; k_y + k_0 v_\ell,\; z) \cdot P(k_x, k_y)$$

### Step 3: Defocus propagation via the angular spectrum method

The effect of defocus at depth $z$ is modeled by the angular spectrum propagation kernel:

$$H_z(k_x, k_y) = P(k_x, k_y) \cdot \exp\!\big(i\, k_z\, z\big)$$

where the axial wavenumber is:

$$k_z = \sqrt{k_0^2 - k_x^2 - k_y^2}$$

This kernel applies a phase shift that depends on depth: in-focus planes get no distortion, while out-of-focus planes accumulate phase that blurs the image.

### Step 4: Measured intensity

The measured intensity for LED $\ell$ at depth $z$ is the squared magnitude of the inverse Fourier transform:

$$I_\ell(x, y, z) = \big|\mathcal{F}^{-1}\!\big\{\tilde{O}_{\text{sub},\ell}(k_x, k_y, z) \cdot H_z(k_x, k_y)\big\}\big|^2$$

### Key insight: synthetic aperture

Each LED captures a different sub-band of the spatial frequency spectrum. Center LEDs (small $u_\ell, v_\ell$) capture low spatial frequencies (coarse structure), while edge LEDs (large $u_\ell, v_\ell$) capture high spatial frequencies (fine detail). By combining all 68 LEDs, we synthesize an effective aperture much larger than the physical pupil, achieving resolution well beyond the objective's native NA = 0.25.

## Method Overview: Implicit Neural Representation (INR)

Instead of storing the complex field $O(x, y, z)$ on a discrete voxel grid, FPM-INR represents it as a continuous function parameterized by two neural networks:

$$O(x, y, z) = \underbrace{A_\theta(x, y, z)}_{\text{amplitude network}} \cdot \exp\!\big(i\, \underbrace{\phi_\theta(x, y, z)}_{\text{phase network}}\big)$$

### Factored architecture

Each network uses a factored design that separates spatial and depth features:

1. **2D spatial feature grid:** A learnable 2D grid of feature vectors, sampled at $(x, y)$ via bilinear interpolation. This captures the in-plane spatial structure.
2. **1D depth feature array:** A learnable 1D array of feature vectors, sampled at $z$ via linear interpolation. This captures how the field varies with depth.
3. **Fusion:** The spatial and depth features are combined by element-wise multiplication.
4. **MLP decoder:** A small multi-layer perceptron maps the fused features to the output (amplitude or phase value).

This factored design is efficient because it avoids the cost of a full 3D feature grid while still capturing 3D structure.

### Training objective

The network parameters $\theta$ are optimized by minimizing the smooth L1 loss between predicted and measured amplitudes:

$$\mathcal{L}(\theta) = \sum_{\ell=1}^{L} \text{SmoothL1}\!\left(\sqrt{I_\ell^{\text{pred}}} - \sqrt{I_\ell^{\text{meas}}}\right)$$

where $I_\ell^{\text{pred}}$ comes from applying the forward model (Steps 1--4 above) to the INR-predicted field, and $\sqrt{\cdot}$ converts intensity to amplitude.

### Training strategy

- **Z-plane sampling:** Each training step selects a batch of z-planes, alternating between uniformly spaced planes and randomly sampled planes. This balances coverage with exploration.
- **Optimizer:** Adam with initial learning rate 0.005, decayed by StepLR (factor 0.95 every 80 steps).
- **Training time:** ~7 minutes on an NVIDIA A40 GPU for 400 epochs.

## Setup

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Add project root to path so we can import src/ modules
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
REF_DIR = os.path.join(PROJECT_ROOT, "evaluation", "reference_outputs")

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Ref dir:      {REF_DIR}")

## Data Exploration

Let's examine the raw FPM measurements. The dataset contains 68 low-resolution intensity images, each captured with a different LED illuminating the sample from a different angle. We'll use the `src/preprocessing` module to load and process the data.

Key things to understand about the data:
- Each LED angle $(u_\ell, v_\ell)$ selects a different sub-band of the sample's spatial frequency spectrum.
- LEDs are sorted by their illumination NA (distance from the optical axis in NA space).
- **Center LEDs** (low NA) illuminate nearly on-axis, capturing low spatial frequencies: these images appear bright with smooth features.
- **Edge LEDs** (high NA) illuminate at steep angles, capturing high spatial frequencies: these images appear dark with fine structural detail visible.

In [ ]:
from src.preprocessing import load_metadata, load_raw_data, compute_optical_params

metadata = load_metadata(DATA_DIR)
raw_data = load_raw_data(DATA_DIR)
optical_params = compute_optical_params(raw_data, metadata)

Isum = optical_params["Isum"]  # (1024, 1024, 68) normalized, sorted by NA
n_leds = Isum.shape[2]
u, v = optical_params["u"], optical_params["v"]

print(f"Measurement stack shape: {Isum.shape}  (H x W x n_LEDs)")
print(f"Number of LEDs:          {n_leds}")
print(f"Wavelength:              {metadata['wavelength_um']*1000:.1f} nm")
print(f"Objective:               {metadata['magnification']}x / {metadata['NA']} NA")
print(f"Camera pixel pitch:      {metadata['pixel_size_um']} um")
print(f"Upsampled grid:          {optical_params['MM']} x {optical_params['NN']} pixels")

### Raw measurements across the NA range

Below we show 6 representative LED measurements spanning from the center (lowest NA, on-axis illumination) to the edge (highest NA, steep-angle illumination). The top row shows the full field of view and the bottom row shows a 256x256 center crop for detail.

**What to look for:**
- LED 0 (lowest NA): Brightest image, dominated by low-frequency (coarse) features. Individual cells are visible but lack fine detail.
- LEDs at intermediate NA: Progressively dimmer, with mid-frequency details becoming visible.
- LED 67 (highest NA): Very dim, showing only high-frequency content (cell edges, fine structure). This is the information that extends the effective resolution beyond the objective's native NA.

In [ ]:
led_indices = [0, 10, 20, 34, 50, 67]  # from center (low NA) to edge (high NA)
NA_illu = np.sqrt(u**2 + v**2)

fig, axes = plt.subplots(2, len(led_indices), figsize=(20, 7))

for col, idx in enumerate(led_indices):
    img = np.sqrt(Isum[:, :, idx])  # show amplitude = sqrt(intensity)
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'LED {idx}\nNA = {NA_illu[idx]:.3f}', fontsize=10)
    axes[0, col].axis('off')

    # Center 256x256 crop
    c = Isum.shape[0] // 2
    crop = img[c-128:c+128, c-128:c+128]
    axes[1, col].imshow(crop, cmap='gray')
    axes[1, col].set_title('Center crop', fontsize=9)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Full FOV', fontsize=11)
axes[1, 0].set_ylabel('256x256 crop', fontsize=11)
plt.suptitle('Raw FPM Measurements (sorted by illumination NA: center -> edge)', fontsize=13)
plt.tight_layout()
plt.show()

### LED illumination pattern in NA space

The scatter plot below shows the position of each LED in numerical aperture (NA) space, where NAx and NAy correspond to the $x$ and $y$ components of the illumination angle. The red dashed circle marks the objective's native NA = 0.256.

**What to look for:**
- LEDs inside the red circle contribute to the **brightfield** image (their illumination falls within the objective's acceptance cone).
- LEDs outside the circle are **darkfield** illuminations --- their direct light does not enter the objective, so only scattered/diffracted light is captured. These carry the high-frequency information that extends resolution.
- The color indicates LED index (sorted by NA), confirming the sorting order used throughout the pipeline.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
sc = ax.scatter(u, v, c=np.arange(n_leds), cmap='viridis', s=40,
                edgecolors='k', linewidths=0.5)
circle = plt.Circle((0, 0), metadata["NA"], fill=False, color='red',
                     linewidth=2, linestyle='--',
                     label=f'Objective NA = {metadata["NA"]}')
ax.add_patch(circle)
ax.set_xlabel('NAx')
ax.set_ylabel('NAy')
ax.set_title('LED Illumination Pattern in NA Space')
ax.set_aspect('equal')
ax.legend(loc='upper right')
plt.colorbar(sc, ax=ax, label='LED index (sorted by NA)')
ax.set_xlim(-0.5, 0.5)
ax.set_ylim(-0.5, 0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Precomputed Results

We load precomputed reconstruction results from `evaluation/reference_outputs/` so the notebook runs in seconds without GPU computation. These were produced by training the FPM-INR model for 400 epochs on an NVIDIA A40 GPU (~7 minutes).

In [ ]:
# Load reference metrics
with open(os.path.join(REF_DIR, "metrics.json")) as f:
    metrics = json.load(f)

# Load all-in-focus images (2048 x 2048)
aif_pred = np.load(os.path.join(REF_DIR, "aif_pred.npy"))
aif_gt = np.load(os.path.join(REF_DIR, "aif_gt.npy"))

# Load per-slice metrics (161 z-positions)
per_slice = np.load(os.path.join(REF_DIR, "per_slice_metrics.npz"))
z_positions = per_slice["z_positions"]
psnr_per_slice = per_slice["psnr_per_slice"]
l2_per_slice = per_slice["l2_per_slice"]
ssim_per_slice = per_slice["ssim_per_slice"]

print("Loaded precomputed results:")
print(f"  All-in-focus images: {aif_pred.shape} (pred), {aif_gt.shape} (GT)")
print(f"  Per-slice metrics:   {len(z_positions)} z-positions from {z_positions[0]:.1f} to {z_positions[-1]:.1f} um")
print()
print("Reference metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## All-in-Focus Comparison

The all-in-focus (AIF) image is computed using the **Normal Variance** method: for each spatial location $(x, y)$, we compute the local variance of the amplitude across a small patch at each z-plane, then select the z-plane with maximum variance (sharpest focus). This produces a single 2D image where every region is in focus, regardless of the sample's tilt.

The primary quantitative metric is the **mean-subtracted L2 error (MSE)**:

$$\text{L2}_{\text{AIF}} = \frac{1}{N_{\text{pix}}} \sum_{x,y} \Big[\big(\hat{I}(x,y) - \overline{\hat{I}}\big) - \big(I_{\text{GT}}(x,y) - \overline{I_{\text{GT}}}\big)\Big]^2$$

where $\hat{I}$ is the predicted AIF image, $I_{\text{GT}}$ is the ground truth AIF image, and the overbar denotes the spatial mean. Subtracting the mean removes any global brightness offset, focusing the comparison on structural content.

**What to look for in the figure below:**
- Left: ground truth all-in-focus image from conventional FPM reconstruction.
- Center: FPM-INR predicted all-in-focus image. It should closely match the GT in structure and contrast.
- Right: error map showing pixel-wise absolute differences after mean subtraction. Hot spots indicate where the reconstruction deviates most --- typically at sharp edges or regions with strong phase gradients.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=150)

axes[0].imshow(aif_gt, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Ground Truth All-in-Focus')
axes[0].axis('off')

axes[1].imshow(aif_pred, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'FPM-INR All-in-Focus\nL2 = {metrics["aif_l2_error"]:.4e}')
axes[1].axis('off')

# Mean-subtracted error map
error_map = np.abs(
    (aif_pred - np.mean(aif_pred)) - (aif_gt - np.mean(aif_gt))
)
im = axes[2].imshow(error_map, cmap='hot', vmin=0, vmax=0.02)
axes[2].set_title('Error Map (mean-subtracted |diff|)')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.suptitle(
    f'All-in-Focus:  L2 = {metrics["aif_l2_error"]:.4e}  |  '
    f'PSNR = {metrics["aif_psnr"]:.1f} dB',
    fontsize=13
)
plt.tight_layout()
plt.show()

## Per-Slice Reconstruction Metrics

The reconstruction is evaluated at 161 ground truth z-positions linearly spaced from $-20$ to $+20$ $\mu$m. At each z-slice, we compare the predicted amplitude image to the ground truth using three metrics:

- **L2 (RMSE):** Root mean squared error between normalized images. Lower is better. Measures pixel-level accuracy.
- **PSNR (Peak Signal-to-Noise Ratio):** $\text{PSNR} = 10 \log_{10}(1 / \text{MSE})$ in dB, where images are normalized to $[0, 1]$. Higher is better. Values above 20 dB indicate good reconstruction quality.
- **SSIM (Structural Similarity Index):** Measures perceptual similarity based on luminance, contrast, and structure. Ranges from 0 to 1, with 1 meaning identical images. SSIM above 0.7 indicates strong structural agreement.

**What to look for:**
- Is reconstruction quality uniform across the depth range, or does it degrade at the extremes ($z = \pm 20$ $\mu$m)?
- Are there systematic patterns (e.g., better at the center, worse at edges) that might indicate limitations of the defocus model?
- How tight is the variance? Small fluctuations suggest the INR generalizes well across depths.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# L2 error vs z
axes[0].plot(z_positions, l2_per_slice, color='tab:red')
axes[0].axhline(np.mean(l2_per_slice), color='gray', linestyle='--', alpha=0.7,
                label=f'Mean = {np.mean(l2_per_slice):.4f}')
axes[0].set_xlabel('z position (um)')
axes[0].set_ylabel('L2 (RMSE)')
axes[0].set_title('Per-Slice L2 Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PSNR vs z
axes[1].plot(z_positions, psnr_per_slice, color='tab:blue')
axes[1].axhline(np.mean(psnr_per_slice), color='gray', linestyle='--', alpha=0.7,
                label=f'Mean = {np.mean(psnr_per_slice):.2f} dB')
axes[1].set_xlabel('z position (um)')
axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('Per-Slice PSNR')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# SSIM vs z
axes[2].plot(z_positions, ssim_per_slice, color='tab:green')
axes[2].axhline(np.mean(ssim_per_slice), color='gray', linestyle='--', alpha=0.7,
                label=f'Mean = {np.mean(ssim_per_slice):.4f}')
axes[2].set_xlabel('z position (um)')
axes[2].set_ylabel('SSIM')
axes[2].set_title('Per-Slice SSIM')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Results Interpretation

### Summary of metrics

| Metric | Our Result | Paper Reference |
|--------|------------|----------------|
| AIF L2 Error (MSE) | 1.33 x 10$^{-3}$ | 1.41 x 10$^{-3}$ |
| AIF PSNR | 28.8 dB | -- |
| Per-slice PSNR (mean +/- std) | 22.8 +/- 0.5 dB | -- |
| Per-slice SSIM (mean +/- std) | 0.775 +/- 0.034 | -- |

### Why INR works well for this problem

The implicit neural representation is a natural fit for FPM 3D reconstruction for several reasons:

1. **Continuous depth representation.** Unlike discrete z-stack methods that reconstruct on a fixed grid, the INR can evaluate the field at any continuous $z$ coordinate. This is critical because the 161 evaluation z-planes are densely spaced (0.25 $\mu$m apart), and a grid-based method would need to store and optimize all slices simultaneously.

2. **Factored architecture efficiency.** By separating spatial (2D) and depth (1D) features and combining them via element-wise product, the network avoids the $O(N^2 \times N_z)$ memory cost of a full 3D grid. The spatial grid captures in-plane structure shared across depths, while the depth array captures how features evolve with defocus.

3. **Implicit regularization.** Neural networks tend to learn smooth, low-frequency features first (spectral bias), which acts as a natural regularizer for this ill-posed inverse problem. This bias is beneficial because biological samples like blood smears have mostly smooth amplitude and phase profiles.

### Connecting metrics to the method

The AIF L2 error of 1.33 x 10$^{-3}$ is slightly better than the paper's reported 1.41 x 10$^{-3}$. This small improvement likely arises from hardware/software differences (PyTorch version, GPU precision, cuDNN autotuning) rather than any algorithmic change. The per-slice PSNR of ~22.8 dB with low standard deviation (0.5 dB) confirms that the INR maintains consistent quality across the full 40 $\mu$m depth range, validating the factored depth representation.

### Limitations

- **Training time:** ~7 minutes on an A40 GPU per sample. Not suitable for real-time imaging.
- **Known illumination angles required:** The method needs accurately calibrated LED positions (NA values). Miscalibration degrades reconstruction quality.
- **Homogeneous medium assumption:** The angular spectrum propagation kernel assumes the sample sits in a uniform medium with known refractive index. Thick or heterogeneous samples would violate this assumption.
- **Single-channel reconstruction:** This demo uses only the green channel ($\lambda = 512$ nm). Multi-channel extension would require separate training or a multi-wavelength forward model.

## Reproducing from Scratch

To run the full FPM-INR training pipeline from scratch (requires GPU, ~7 min on A40), uncomment and run the cell below. This will:
1. Load and preprocess the raw FPM measurements
2. Build the forward model and INR network
3. Train for 400 epochs using the smooth L1 loss
4. Save results to `evaluation/reference_outputs/`

In [ ]:
# === Uncomment to run the full pipeline (~7 min on A40 GPU) ===
#
# import torch
# from src.preprocessing import prepare_data, load_ground_truth
# from src.physics_model import FPMForwardModel
# from src.solvers import FullModel, save_model_with_required_grad
# from src.solvers import FPMINRSolver
# from src.visualization import all_in_focus_normal_variance
# from src.visualization import compute_metrics, compute_allfocus_l2
#
# torch.manual_seed(0)
# device = "cuda:0"
#
# # Step 1: Load and preprocess data
# data = prepare_data(data_dir=DATA_DIR, device=device)
# metadata_full = data["metadata"]
# training_cfg = metadata_full["training"]
#
# # Step 2: Build forward model
# forward_model = FPMForwardModel(
#     Pupil0=data["pupil_data"]["Pupil0"],
#     kzz=data["pupil_data"]["kzz"],
#     ledpos_true=data["optical_params"]["ledpos_true"],
#     M=data["optical_params"]["M"],
#     N=data["optical_params"]["N"],
#     MAGimg=data["optical_params"]["MAGimg"],
# )
#
# # Step 3: Build INR network
# model = FullModel(
#     w=data["optical_params"]["MM"],
#     h=data["optical_params"]["MM"],
#     num_feats=training_cfg["num_feats"],
#     x_mode=metadata_full["num_modes"],
#     y_mode=metadata_full["num_modes"],
#     z_min=data["z_params"]["z_min"],
#     z_max=data["z_params"]["z_max"],
#     ds_factor=1,
#     use_layernorm=False,
# ).to(device)
#
# # Step 4: Train
# solver = FPMINRSolver(num_epochs=training_cfg["num_epochs"])
# results = solver.train(model, forward_model, data["Isum"], data["z_params"], device)
# print(f"Training complete. Final PSNR: {results['final_psnr']:.2f} dB")